In [1]:
import numpy as np
import pandas as pd
import scanpy as sc

In [2]:
import session_info
session_info.show()

/home/kk837/.conda/envs/generic_env/lib/python3.10/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  mod_version = _find_version(mod.__version__)
/home/kk837/.conda/envs/generic_env/lib/python3.10/site-packages/session_info/main.py:213: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  mod_version = _find_version(mod.__version__)


In [14]:
import importlib.util
import sys
spec = importlib.util.spec_from_file_location("module.name", "/rfs/project/rfs-iCNyzSAaucw/kk837/function/python/utils.py")
utils = importlib.util.module_from_spec(spec)
sys.modules["module.name"] = utils
spec.loader.exec_module(utils)

/rfs/project/rfs-iCNyzSAaucw/kk837/function/python/utils.py:356: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if sc.__version__.startswith("1.4"):


# Variables

In [3]:
data_object_dir = '/rfs/project/rfs-iCNyzSAaucw/kk837/data_objects/Foetal/VisiumHD/Revision_Oct2025'
!ls {data_object_dir}

all_b2c_cells_filtered_celltype-sel.h5ad
all_b2c_cells_filtered_raw.h5ad
all_b2c_cells_raw.h5ad
archive_17Nov2025
epicardium-mixture_b2c_cells_filtered_raw.h5ad
erythrocytes_b2c_cells_filtered_raw.h5ad
scVI
unclassified-mixture_b2c_cells_filtered_raw.h5ad


# Read in HD adata

In [4]:
adata = sc.read_h5ad(f'{data_object_dir}/all_b2c_cells_filtered_celltype-sel.h5ad')
adata

AnnData object with n_obs × n_vars = 217420 × 18085
    obs: 'object_id', 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'in_tissue_manual', 'library', 'donor_section_ID', 'per-frame_donorIDs', 'donor', 'CS', 'est_CS', 'GA', 'PCW', 'section_ID', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'library_donor', 'n_genes', 'coarse_grain_pre', 'coarse_grain_pre_2'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'PCW_colors', 'coarse_grain_pre_2_colors', 'coarse_grain_pre_colors', 'donor_colors', 'library_colors', 'neighbors', 'spatial', 'umap'
    obsm: 'spatial', 'spatial_cropped_150_buffer'
    obsp: 'connectivities', 'dist

In [6]:
# subset already classified cells
adata = adata[adata.obs['coarse_grain_pre_2'].isin(['unclassified_mixture',
                                                     'unclassified_mixture_Mesenchymal-LEC-Immune',
                                                     'unclassified_mixture_aCM-Myeloid',
                                                     'unclassified_mixture_aCM-epicardium',
                                                     'unclassified_mixture_aCM-vasculature',
                                                     'unclassified_mixture_vCM-epicardium',
                                                     'unclassified_mixture_vCM-macrophage',
                                                     'unclassified_mixture_vCM-vasculature'])==False]
adata.obs['coarse_grain_pre_2'].value_counts()

coarse_grain_pre_2
VentricularCardiomyocytes    106519
MesenchymalCells              38029
AtrialCardiomyocytes          31838
LymphaticEndothelialCells      2169
EndothelialCells               2065
NeuralCells                    1851
EpicardialCells                1408
MyeloidCells                   1157
Name: count, dtype: int64

# Read in single-cell/nuc RNA-seq data

In [7]:
adata_scn = sc.read_h5ad('/rfs/project/rfs-iCNyzSAaucw/kk837/data_objects/Foetal/RNA/Feb28ObjectRaw_finegrain_updated.h5ad')
adata_scn

AnnData object with n_obs × n_vars = 297473 × 36601
    obs: 'latent_RT_efficiency', 'latent_cell_probability', 'latent_scale', 'sangerID', 'combinedID', 'donor', 'region', 'age', 'facility', 'cell_or_nuclei', 'modality', 'kit_10x', 'scrublet_score', 'doublet_pval', 'doublet_bh_pval', 'n_genes', 'n_counts', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'HB_score', 'donor_by_library-prep', 'multiplexed', 'SOC | status', 'SOC | log_prob_singleton', 'SOC | log_prob_doublet', 'batch_key', '_scvi_batch', 'FACSgate', 'fine_grain', 'mid_grain', 'coarse_grain', 'sex', 'week', 'trimester', 'heart_or_greatvessels', 'cycling', 'S_score', 'G2M_score', 'phase', '_scvi_labels', 'stress_score', 'hb1_score'
    var: 'gene_name_scRNA-0', 'gene_id'
    uns: 'FACSgate_colors', '_scvi_manager_uuid', '_scvi_uuid', 'age_colors', 'cell_or_nucl

In [11]:
# modify mid_grain (to separate CCS cells)
acm_obsnames = adata_scn.obs_names[adata_scn.obs['fine_grain'].isin(['AtrialCardiomyocytesCycling',
                                                                     'AtrialCardiomyocytesLeft',
                                                                     'AtrialCardiomyocytesRight',
                                                                     'AtrioventricularNodePacemakerCells',
                                                                    'SinoatrialNodePacemakerCells'])]
vcm_obsnames = adata_scn.obs_names[adata_scn.obs['fine_grain'].isin(['VentricularCardiomyocytesCycling',
                                                             'VentricularCardiomyocytesLeftCompact',
                                                             'VentricularCardiomyocytesLeftTrabeculated',
                                                             'VentricularCardiomyocytesRightCompact',
                                                             'VentricularCardiomyocytesRightTrabeculated',
                                                             'VentricularConductionSystemDistal',
                                                             'VentricularConductionSystemProximal'])]

adata_scn.obs['mid_grain_mod'] = adata_scn.obs['mid_grain'].astype('str').copy()
adata_scn.obs.loc[acm_obsnames,'mid_grain_mod'] = 'AtrialCardiomyocytes'
adata_scn.obs.loc[vcm_obsnames,'mid_grain_mod'] = 'VentricularCardiomyocytes'
adata_scn.obs['mid_grain_mod'].value_counts()

mid_grain_mod
Fibroblasts                    102791
VentricularCardiomyocytes       53268
MuralCells                      28497
AtrialCardiomyocytes            21616
LymphoidCells                   21084
MyeloidCells                    19962
PericardialCells                16811
BloodVesselEndothelialCells     11352
EndocardialCells                 9346
EpicardialCells                  4186
Glia                             3541
Neurons                          2608
LymphaticEndothelialCells        2411
Name: count, dtype: int64

# Subsample scnRNA-seq data based on cell number of VisiumHD data and number of fine_grain cell types

- For each cell type label, subsample scnRNA-seq data to be same cell number as the VisiumHD classified data or smaller number
- For each cell type label, split the VisiumHD cell number into the number of fine_grain cell types (in scnRNAseq). That number will be the max_n for subsampling per fine_grain.
- For cell types which doesn't exist in VisiumHD data, keep all the cells

In [16]:
set(adata.obs['coarse_grain_pre_2'])

{'AtrialCardiomyocytes',
 'EndothelialCells',
 'EpicardialCells',
 'LymphaticEndothelialCells',
 'MesenchymalCells',
 'MyeloidCells',
 'NeuralCells',
 'VentricularCardiomyocytes'}

In [18]:
set(adata_scn.obs['coarse_grain'])

{'Cardiomyocytes',
 'Endothelium',
 'Epicardium',
 'Leukocytes',
 'Mesenchymal',
 'Neural'}

In [19]:
set(adata_scn.obs['mid_grain'])

{'AtrialCardiomyocytes',
 'BloodVesselEndothelialCells',
 'CardiacConductionSystem',
 'EndocardialCells',
 'EpicardialCells',
 'Fibroblasts',
 'Glia',
 'LymphaticEndothelialCells',
 'LymphoidCells',
 'MuralCells',
 'MyeloidCells',
 'Neurons',
 'PericardialCells',
 'VentricularCardiomyocytes'}

In [28]:
celltype_dict = {
    'AtrialCardiomyocytes':['mid_grain_mod',['AtrialCardiomyocytes']],
     'EndothelialCells':['mid_grain_mod',['BloodVesselEndothelialCells','EndocardialCells']],
     'EpicardialCells':['coarse_grain',['Epicardium']],
     'LymphaticEndothelialCells':['mid_grain_mod',['LymphaticEndothelialCells']],
     'MesenchymalCells':['coarse_grain',['Mesenchymal']],
     'MyeloidCells':['mid_grain_mod',['MyeloidCells']],
     'NeuralCells':['coarse_grain',['Neural']],
     'VentricularCardiomyocytes':['mid_grain_mod',['VentricularCardiomyocytes']],
    'not_found':['mid_grain_mod',['LymphoidCells']],
}

In [21]:
set(adata.obs['coarse_grain_pre_2']) - set(celltype_dict.keys())

set()

In [29]:
adata_ref_list = []
for celltype,subset_info in celltype_dict.items():
    if celltype!='not_found':
        print(f'### {celltype} ###')
        hd_sub = adata[adata.obs['coarse_grain_pre_2']==celltype]
        scn_sub = adata_scn[adata_scn.obs[subset_info[0]].isin(subset_info[1])]
        max_n = round(hd_sub.shape[0]/len(set(scn_sub.obs['fine_grain'])))
        print(f'max_n for subsampling per fine-grain: {max_n}')
        # subsampling
        scn_sub = utils.sctk_subsample(scn_sub,fraction=1,groupby='fine_grain',max_n=max_n)
        print('value_counts of scnRNAseq, per fine-grain')
        print(scn_sub.obs['fine_grain'].value_counts())
        # concatenate
        hd_sub.obs['reference_type'] = 'VisiumHD'
        scn_sub.obs['reference_type'] = 'scnRNAseq'
        ad_concat = hd_sub.concatenate(scn_sub, 
                                       join='inner', 
                                       batch_key=None,
                                       index_unique=None
                                      )
        ad_concat.obs['reference_celltype'] = celltype
        print('value_counts of reference type')
        print(ad_concat.obs['reference_type'].value_counts())
        adata_ref_list.append(ad_concat)
        del hd_sub,scn_sub,max_n,ad_concat
    else:
        print(f'### {subset_info} ###')
        scn_sub = adata_scn[adata_scn.obs[subset_info[0]].isin(subset_info[1])]
        max_n = 1000
        print(f'max_n for subsampling per fine-grain: {max_n}')
        scn_sub = utils.sctk_subsample(scn_sub,fraction=1,groupby='fine_grain',max_n=max_n)
        print('value_counts of scnRNAseq, per fine-grain')
        print(scn_sub.obs['fine_grain'].value_counts())
        scn_sub.obs['reference_type'] = 'scnRNAseq'
        scn_sub.obs['reference_celltype'] = '_'.join(subset_info[1])
        print(scn_sub.obs['reference_type'].value_counts())
        adata_ref_list.append(scn_sub)
        del scn_sub,max_n
    print('')

### AtrialCardiomyocytes ###
max_n for subsampling per fine-grain: 6368
value_counts of scnRNAseq, per fine-grain
fine_grain
AtrialCardiomyocytesLeft              6368
AtrialCardiomyocytesRight             6368
AtrialCardiomyocytesCycling           2396
SinoatrialNodePacemakerCells           981
AtrioventricularNodePacemakerCells     937
Name: count, dtype: int64


/tmp/ipykernel_434091/1800732968.py:14: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  hd_sub.obs['reference_type'] = 'VisiumHD'
/tmp/ipykernel_434091/1800732968.py:16: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  ad_concat = hd_sub.concatenate(scn_sub,


value_counts of reference type
reference_type
VisiumHD     31838
scnRNAseq    17050
Name: count, dtype: int64

### EndothelialCells ###
max_n for subsampling per fine-grain: 258
value_counts of scnRNAseq, per fine-grain
fine_grain
GreatVesselArterialEndothelialCells    258
GreatVesselVenousEndothelialCells      258
CoronaryArterialEndothelialCells       258
CoronaryVenousEndothelialCells         258
CoronaryCapillaryEndothelialCells      258
EndocardialCells                       258
EndocardialCushionCells                258
ValveEndothelialCells                  258
Name: count, dtype: int64


/tmp/ipykernel_434091/1800732968.py:14: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  hd_sub.obs['reference_type'] = 'VisiumHD'
/tmp/ipykernel_434091/1800732968.py:16: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  ad_concat = hd_sub.concatenate(scn_sub,


value_counts of reference type
reference_type
VisiumHD     2065
scnRNAseq    2064
Name: count, dtype: int64

### EpicardialCells ###
max_n for subsampling per fine-grain: 704
value_counts of scnRNAseq, per fine-grain
fine_grain
MesothelialEpicardialCells    704
EpicardiumDerivedCells        704
Name: count, dtype: int64


...

value_counts of reference type
reference_type
VisiumHD     1157
scnRNAseq    1152
Name: count, dtype: int64

### NeuralCells ###
max_n for subsampling per fine-grain: 308
value_counts of scnRNAseq, per fine-grain
fine_grain
NeuronPrecursors          308
SympatheticNeurons        308
SchwannCellPrecursors     308
ParasympatheticNeurons    308
SchwannCells              308
ChromaffinCells           228
Name: count, dtype: int64


/tmp/ipykernel_434091/1800732968.py:14: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  hd_sub.obs['reference_type'] = 'VisiumHD'
/tmp/ipykernel_434091/1800732968.py:16: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  ad_concat = hd_sub.concatenate(scn_sub,


value_counts of reference type
reference_type
VisiumHD     1851
scnRNAseq    1768
Name: count, dtype: int64

### VentricularCardiomyocytes ###
max_n for subsampling per fine-grain: 15217
value_counts of scnRNAseq, per fine-grain
fine_grain
VentricularCardiomyocytesLeftCompact          15217
VentricularCardiomyocytesRightCompact         15217
VentricularCardiomyocytesCycling               8195
VentricularCardiomyocytesLeftTrabeculated      6997
VentricularCardiomyocytesRightTrabeculated     3101
VentricularConductionSystemProximal            2209
VentricularConductionSystemDistal               960
Name: count, dtype: int64


/tmp/ipykernel_434091/1800732968.py:14: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  hd_sub.obs['reference_type'] = 'VisiumHD'
/tmp/ipykernel_434091/1800732968.py:16: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  ad_concat = hd_sub.concatenate(scn_sub,


value_counts of reference type
reference_type
VisiumHD     106519
scnRNAseq     51896
Name: count, dtype: int64

### ['mid_grain_mod', ['LymphoidCells']] ###
max_n for subsampling per fine-grain: 1000
value_counts of scnRNAseq, per fine-grain
fine_grain
TCellsCD4pos           1000
TCellsCD8pos           1000
TregsCD4pos            1000
ProBCells              1000
BCells                 1000
BCellsMS4A1pos         1000
NaturalKillerCells     1000
InnateLymphoidCells    1000
Name: count, dtype: int64
reference_type
scnRNAseq    8000
Name: count, dtype: int64



In [30]:
len(adata_ref_list)

9

In [31]:
# concatenate
adata_ref = adata_ref_list[0].concatenate(adata_ref_list[1:], 
                                       join='inner', 
                                       batch_key=None,
                                       index_unique=None
                                      )
ctab = pd.crosstab(adata_ref.obs['reference_celltype'],adata_ref.obs['reference_type'])
print(ctab.shape)
ctab

/tmp/ipykernel_434091/3795423091.py:2: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_ref = adata_ref_list[0].concatenate(adata_ref_list[1:],


(9, 2)


reference_type,VisiumHD,scnRNAseq
reference_celltype,,
AtrialCardiomyocytes,31838,17050
EndothelialCells,2065,2064
EpicardialCells,1408,1408
LymphaticEndothelialCells,2169,2169
LymphoidCells,0,8000
MesenchymalCells,38029,32159
MyeloidCells,1157,1152
NeuralCells,1851,1768
VentricularCardiomyocytes,106519,51896


# Save

In [33]:
# save
print(adata_ref.X.data[:10])
print(adata_ref.shape)
adata_ref.write(f'{data_object_dir}/TACCO/reference_raw.h5ad')

[2. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
(302702, 18085)
